In [ ]:
# Train Setup

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if device == 'cuda' and torch.cuda.is_bf16_supported():
    dtype = torch.bfloat16
elif device == 'cuda':
    dtype = torch.float16
else:
    dtype = torch.float32

print(f"{device=}\n")
print(f"{dtype=}\n")

# config
lr = 4e-3
train_iter = 10000
batch_size = 64
dim = 64
n_heads = 4
head_dims = dim // n_heads
n_layers = 4
mlp_dim = 1024
block_size = 256
eval_batches = 10
show_plots = False

with open("input.txt", "r") as f:
    text = f.read()

# char tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"{vocab_size=}\n")
print(f"{dim=}\n")
print(f"{n_layers=}\n")

# split data: 80% train, 10% val, 10% test
tokens = torch.tensor(encode(text), device=device)
n = len(tokens)
train_tokens = tokens[:int(0.8 * n)]
val_tokens = tokens[int(0.8 * n):int(0.9 * n)]
test_tokens = tokens[int(0.9 * n):]

print(f"Train = {len(train_tokens)}")
print(f"Val = {len(val_tokens)}")
print(f"Test = {len(test_tokens)}\n")

# sliding windows per split
train_seqs = train_tokens.unfold(0, block_size + 1, 1)
val_seqs = val_tokens.unfold(0, block_size + 1, 1)
test_seqs = test_tokens.unfold(0, block_size + 1, 1)

emb = torch.randn(vocab_size, dim, device=device, dtype=dtype) * 0.02
pos = torch.randn(block_size, dim, device=device, dtype=dtype) * 0.02

# one set of weights per layer
init_scale = 0.1 / (2 ** 0.5)
wq = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
wk = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
wv = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
wo = [torch.randn(dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
w1 = [torch.randn(dim, mlp_dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]
w2 = [torch.randn(mlp_dim, dim, device=device, dtype=dtype) * init_scale for _ in range(n_layers)]

params = [emb, pos] + wq + wk + wv + wo + w1 + w2
for p in params:
    p.requires_grad = True

def rmsnorm(x, eps=1e-5):
    return x / ((x ** 2).mean(dim=-1, keepdim=True).sqrt() + eps)

def forward(x, stop_after=None, steer_layer=None, steer_vec=None, return_attn=False):
    B, T = x.shape
    x = emb[x] + pos[:T]
    mask = torch.tril(torch.ones(T, T, device=device, dtype=torch.bool))
    attn_weights = [] if return_attn else None

    last_layer = stop_after if stop_after is not None else n_layers - 1
    for layer in range(last_layer + 1):
        nx = rmsnorm(x)

        q, k, v = nx @ wq[layer], nx @ wk[layer], nx @ wv[layer]
        q = q.view(B,T,n_heads,head_dims).transpose(1,2)
        k = k.view(B,T,n_heads,head_dims).transpose(1,2)
        v = v.view(B,T,n_heads,head_dims).transpose(1,2)

        scores = q @ k.transpose(-2, -1) / (head_dims ** 0.5)
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        if return_attn:
            attn_weights.append(attn.detach())
        attn_out = (attn @ v).transpose(1,2).contiguous().view(B, T, dim)
        x = x + attn_out @ wo[layer]

        nx = rmsnorm(x)
        x = x + (nx @ w1[layer]).relu() @ w2[layer]

        if layer == steer_layer and steer_vec is not None:
            x[0, -1] += steer_vec

    if stop_after is not None:
        return x

    x = rmsnorm(x)
    x = x @ emb.T
    if return_attn:
        return x, attn_weights # attn_weights[layer] = (B, n_heads, T, T)
    return x

@torch.no_grad()
def eval_loss(seqs, n_batches=eval_batches):
    total = 0.0
    for _ in range(n_batches):
        idx = torch.randint(0, seqs.shape[0], (batch_size,))
        x, y = seqs[idx, :-1], seqs[idx, 1:]
        total += F.cross_entropy(forward(x).view(-1, vocab_size), y.view(-1)).item()
    return total / n_batches

forward_raw = forward # uncompiled for inference, SAE, steering
if hasattr(torch, 'compile'):
    forward = torch.compile(forward)
    print("Using torch.compile")

opt = torch.optim.Adam(params, lr=lr, fused=(device == 'cuda'))

In [ ]:
# Cell 2 — Save / Load Weights
def save_weights(path="shakesp_minimal_transformer_weights.pt"):
    torch.save({
        'emb': emb, 'pos': pos,
        'wq': wq, 'wk': wk, 'wv': wv, 'wo': wo,
        'w1': w1, 'w2': w2,
    }, path)
    print(f"Saved to {path}")

def load_weights(path="shakesp_minimal_transformer_weights.pt"):
    global emb, pos, wq, wk, wv, wo, w1, w2, params
    ckpt = torch.load(path, map_location=device, weights_only=True)
    emb, pos = ckpt['emb'], ckpt['pos']
    wq, wk, wv, wo = ckpt['wq'], ckpt['wk'], ckpt['wv'], ckpt['wo']
    w1, w2 = ckpt['w1'], ckpt['w2']
    params = [emb, pos] + wq + wk + wv + wo + w1 + w2
    for p in params:
        p.requires_grad = True
    print(f"Loaded from {path}")

# save_weights()
# load_weights()

Loaded from shakesp_minimal_transformer_weights.pt


In [33]:
# Cell 3 — Generate
ctx = "Firs"
tokens = encode(ctx)

for i in range(1):
    x = torch.tensor([tokens[-block_size:]], device=device)
    logits = forward_raw(x)
    probs = F.softmax(logits[0, -1]/0.8, dim=-1)
    next_token = probs.argmax().item()
    # next_token = torch.multinomial(probs, 1).item()
    tokens.append(next_token)
print(decode(tokens))

First


In [ ]:
# Cell 4 — Enable Plots
show_plots = True
plot_every = 1000 # plot embedding map every N iterations
train_losses = [] # (step, loss) tuples
val_losses = []   # (step, loss) tuples

def plot_embeddings(step, loss_val):
    E = emb.detach().float().cpu()
    E = E - E.mean(dim=0)
    U, S, V = torch.svd(E)
    coords = E @ V[:, :2]
    plt.figure(figsize=(8, 8))
    plt.scatter(coords[:, 0].numpy(), coords[:, 1].numpy(), s=20)
    for i, c in itos.items():
        label = repr(c) if c in (' ', '\n', '\t') else c
        plt.annotate(label, (coords[i, 0].item(), coords[i, 1].item()), fontsize=11)
    plt.title(f"Step {step} | Loss {loss_val:.2f}")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Train Loop

eval_every = 10
for i in range(train_iter):
    idx = torch.randint(0, train_seqs.shape[0], (batch_size,))
    x, y = train_seqs[idx, :-1], train_seqs[idx, 1:]

    logits = forward(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    opt.zero_grad()
    loss.backward()
    opt.step()

    if i % eval_every == 0:
        train_l = loss.item()
        val_l = eval_loss(val_seqs)
        print(f"{i}: train={train_l:.2f}, val={val_l:.2f}")
        if show_plots:
            train_losses.append((i, train_l))
            val_losses.append((i, val_l))

    if show_plots and i % plot_every == 0:
        plot_embeddings(i, loss.item())

test_loss = eval_loss(test_seqs, n_batches=50)
print(f"Test loss: {test_loss:.2f}")

if show_plots:
    steps_t, losses_t = zip(*train_losses)
    steps_v, losses_v = zip(*val_losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps_t, losses_t, label='train')
    plt.plot(steps_v, losses_v, label='val')
    plt.axhline(y=test_loss, color='r', linestyle='--', label=f'test={test_loss:.2f}')
    plt.xlabel('step'); plt.ylabel('loss'); plt.legend()
    plt.title('Train / Val / Test Loss')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Cell 6 — Attention Heatmap
import numpy as np

text_input = "Firs"
tok = torch.tensor([encode(text_input)], device=device)
T = tok.shape[1]

_, attn_weights = forward_raw(tok, return_attn=True) # attn_weights[layer] = (B, n_heads, T, T)

fig, axes = plt.subplots(n_layers, n_heads, figsize=(3 * n_heads, 3 * n_layers))
if n_layers == 1 and n_heads == 1:
    axes = np.array([[axes]])
elif n_layers == 1:
    axes = axes[np.newaxis, :]
elif n_heads == 1:
    axes = axes[:, np.newaxis]

labels = [repr(c) if c in (' ', '\n', '\t') else c for c in text_input]

for layer in range(n_layers):
    for h in range(n_heads):
        ax = axes[layer, h]
        w = attn_weights[layer][0, h].float().cpu().numpy()
        ax.imshow(w, cmap='viridis', vmin=0, vmax=1)
        ax.set_xticks(range(T)); ax.set_xticklabels(labels, fontsize=8)
        ax.set_yticks(range(T)); ax.set_yticklabels(labels, fontsize=8)
        ax.set_title(f'L{layer} H{h}', fontsize=9)

plt.suptitle(f'Attention weights — "{text_input}"', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# SAE + Steering

sae_dim = 256
sae_layer = 3
sae_lr = 1e-3
sae_steps = 3000
sae_l1_lambda = 0.02

sae_enc = torch.randn(dim, sae_dim, device=device, dtype=torch.float32) * 0.01
sae_dec = torch.randn(sae_dim, dim, device=device, dtype=torch.float32) * 0.01
sae_enc_b = torch.zeros(sae_dim, device=device, dtype=torch.float32)
sae_dec_b = torch.zeros(dim, device=device, dtype=torch.float32)
sae_params = [sae_enc, sae_dec, sae_enc_b, sae_dec_b]
for p in sae_params:
    p.requires_grad = True

char_label = lambda c: {' ': '<space>', '\n': '<newline>', '\t': '<tab>'}.get(c, c)
sae_encode = lambda x: F.relu(x @ sae_enc + sae_enc_b)

# Collect activations at sae_layer
@torch.no_grad()
def collect_activations(seqs, n_batches=10):
    all_acts, all_toks, all_targets = [], [], []
    for _ in range(n_batches):
        idx = torch.randint(0, seqs.shape[0], (batch_size,))
        x_in, x_tgt = seqs[idx, :-1], seqs[idx, 1:]
        all_acts.append(forward_raw(x_in, stop_after=sae_layer).view(-1, dim).float())
        all_toks.append(x_in.view(-1))
        all_targets.append(x_tgt.view(-1))
    return torch.cat(all_acts), torch.cat(all_toks), torch.cat(all_targets)

print(f"Collecting activations from layer {sae_layer}...")
acts, toks, targets = collect_activations(train_seqs)
print(f"Collected {acts.shape[0]} vectors, shape {acts.shape}")

# Train SAE
sae_opt = torch.optim.Adam(sae_params, lr=sae_lr)
for step in range(sae_steps):
    batch = acts[torch.randint(0, acts.shape[0], (512,))]
    h = sae_encode(batch)
    recon_loss = (batch - (h @ sae_dec + sae_dec_b)).pow(2).mean()
    l1_loss = h.abs().mean()
    loss = recon_loss + sae_l1_lambda * l1_loss
    sae_opt.zero_grad(); loss.backward(); sae_opt.step()
    if step % 500 == 0:
        print(f"  step {step}: recon={recon_loss.item():.4f}, l1={l1_loss.item():.4f}, alive={(h > 0).float().mean().item():.2%}")
print("SAE done.")

# Encode all activations once, reuse everywhere
h_all = sae_encode(acts).detach()
fire_rate = (h_all > 0).float().mean(dim=0)

def rank_features(top_n=15):
    valid = ((fire_rate > 0.01) & (fire_rate < 0.5)).nonzero().squeeze(-1)
    if len(valid) == 0: return None
    return valid[h_all[:, valid].mean(dim=0).argsort(descending=True)][:top_n]

# Top activating contexts per feature
ranked = rank_features(10)
if ranked is not None:
    for feat_idx in ranked:
        top_idx = h_all[:, feat_idx].argsort(descending=True)[:8]
        print(f"\n── Feature {feat_idx.item()} (fires {fire_rate[feat_idx].item():.1%}) ──")
        for j in top_idx:
            print(f"  '{char_label(itos[toks[j].item()])}' act={h_all[j, feat_idx].item():.2f}")

# Find which feature most activates when predicting a given char
def find_feature_for_predicting(char):
    mask = (targets == stoi[char])
    if not mask.any(): return None
    feat = h_all[mask].mean(dim=0).argmax().item()
    print(f"Prediction feature for '{char_label(char)}': F{feat}")
    return feat

comma_pred = find_feature_for_predicting(',')
space_pred = find_feature_for_predicting(' ')
newline_pred = find_feature_for_predicting('\n')

# Heatmap: mean SAE activation per feature per char
def plot_heatmap(token_ids, title, top_n=15):
    ranked = rank_features(top_n)
    if ranked is None: return
    grid = torch.zeros(len(ranked), vocab_size)
    for i, f in enumerate(ranked):
        for c in range(vocab_size):
            m = (token_ids == c)
            if m.any(): grid[i, c] = h_all[m, f].mean().item()
    plt.figure(figsize=(14, max(3, len(ranked) * 0.4)))
    plt.imshow(grid.numpy(), aspect='auto', cmap='viridis')
    plt.xticks(range(vocab_size), [char_label(itos[c]) for c in range(vocab_size)], fontsize=7, rotation=90)
    plt.yticks(range(len(ranked)), [f"F{f.item()}" for f in ranked], fontsize=8)
    plt.colorbar(label='mean activation'); plt.title(title); plt.tight_layout(); plt.show()

plot_heatmap(toks, f'DETECTION — feature fires when char IS current token (layer {sae_layer})')
plot_heatmap(targets, f'PREDICTION — feature fires when char is NEXT token (layer {sae_layer})')

# Steering: inject sae_dec direction at sae_layer, compare normal vs steered top-5
@torch.no_grad()
def steer(text, sae_dec_i, strength):
    tok = torch.tensor([encode(text)], device=device)
    top5 = lambda lg: [(char_label(itos[i.item()]), f"{v.item():.1%}") for v, i in zip(*F.softmax(lg, dim=-1).topk(5))]
    normal_logits = forward_raw(tok)[0, -1]
    steered_logits = forward_raw(tok, steer_layer=sae_layer, steer_vec=sae_dec_i.float() * strength)[0, -1]
    print(f'Input: "{text}"')
    print(f"  Normal:  {top5(normal_logits)}")
    print(f"  Steered: {top5(steered_logits)}")

In [76]:
# Cell 8 — Steer Test
# space_pred = find_feature_for_predicting('q')
for i in range(10):
    print(i)
    steer("To be or not to be, that is the q", sae_dec[i], 2000.0)


0
Input: "To be or not to be, that is the q"
  Normal top-5:  [('u', '100.0%'), ('o', '0.0%'), ('l', '0.0%'), ('m', '0.0%'), ('W', '0.0%')]
  Steered top-5: [('c', '38.3%'), ('s', '25.1%'), ('k', '13.0%'), ('u', '3.4%'), ('q', '1.9%')]
1
Input: "To be or not to be, that is the q"
  Normal top-5:  [('u', '100.0%'), ('o', '0.0%'), ('l', '0.0%'), ('m', '0.0%'), ('W', '0.0%')]
  Steered top-5: [('<space>', '87.0%'), (',', '10.9%'), ('.', '0.3%'), ('?', '0.3%'), ('y', '0.3%')]
2
Input: "To be or not to be, that is the q"
  Normal top-5:  [('u', '100.0%'), ('o', '0.0%'), ('l', '0.0%'), ('m', '0.0%'), ('W', '0.0%')]
  Steered top-5: [('I', '98.6%'), ("'", '0.8%'), ('E', '0.3%'), ('y', '0.1%'), ('m', '0.1%')]
3
Input: "To be or not to be, that is the q"
  Normal top-5:  [('u', '100.0%'), ('o', '0.0%'), ('l', '0.0%'), ('m', '0.0%'), ('W', '0.0%')]
  Steered top-5: [('T', '31.3%'), ('k', '30.4%'), ('G', '21.2%'), ('w', '9.1%'), ('t', '2.9%')]
4
Input: "To be or not to be, that is the q"
  Normal